# OpenTools evaluation, contribution, MCP, and DSPy demo

This notebook exercises the implemented paths with real local code. It does not mock tool, MCP, evaluation, or LLM results. Optional integrations are explicitly reported as `SKIPPED` when their packages are unavailable.

From the repository root, install the full demo environment if needed:

```bash
python -m pip install -e '.[mcp,dspy,webui]'
```

In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
REPO = next((p for p in candidates if (p / 'src/opentools').is_dir()), None)
if REPO is None:
    raise RuntimeError('Run this notebook from within the OpenTools repository.')
sys.path.insert(0, str(REPO / 'src'))
print(f'Repository: {REPO}')


## 1. Static risk inspection

Static inspection parses source without importing or executing the tool. Its findings are review signals, not a security guarantee.

In [ ]:
from opentools.evaluation import inspect_source

calculator_source = REPO / 'src/opentools/tools/calculator/tool.py'
inspection = inspect_source(calculator_source)
print({
    'risk_level': inspection['risk_level'],
    'files_scanned': inspection['files_scanned'],
    'observed_credentials': inspection['observed_credentials'],
    'finding_kinds': sorted({item['kind'] for item in inspection['findings']}),
})
assert inspection['files_scanned'] == 1


## 2. Build the evaluation inventory in memory

This reads real tool sources and existing result artifacts without executing every tool. The scheduled CI workflow uses the related refresh command to propose table changes through a pull request.

In [ ]:
from collections import Counter
from opentools.inventory import build_index

tools_root = REPO / 'src/opentools/tools'
inventory = build_index(tools_root)
statuses = Counter(record['evaluation_status'] for record in inventory['tools'].values())
print(f"Discovered tools: {len(inventory['tools'])}")
print(f'Evaluation statuses: {dict(statuses)}')
print('Calculator:', inventory['tools']['Calculator_Tool'])


## 3. Convert and execute a real submitted function

Conversion itself does not execute the submission and marks functional evaluation as `not_run`. This cell then deliberately imports the generated bundle and calls it, demonstrating the generated wrapper with an actual result.

In [ ]:
import importlib.util
import inspect
import tempfile
from opentools.conversion import convert_submission
from opentools.core.base import BaseTool

with tempfile.TemporaryDirectory() as directory:
    root = Path(directory)
    source = root / 'submitted.py'
    readme = root / 'README.md'
    source.write_text(
        'def multiply(left: int, right: int) -> int:\n'
        '    \"\"\"Multiply two integers.\"\"\"\n'
        '    return left * right\n',
        encoding='utf-8',
    )
    readme.write_text('# Local multiplier\n', encoding='utf-8')
    converted = convert_submission(
        source, readme, root / 'contributions',
        name='Local Multiplier', license_name='Apache-2.0',
    )
    print('Conversion:', converted['manifest']['conversion_status'])
    print('Evaluation:', converted['manifest']['functional_evaluation'])
    print('Risk:', converted['manifest']['risk']['risk_level'])

    tool_file = Path(converted['bundle']) / 'tool.py'
    spec = importlib.util.spec_from_file_location('local_multiplier_bundle', tool_file)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    tool_class = next(
        value for _, value in inspect.getmembers(module, inspect.isclass)
        if value is not BaseTool and issubclass(value, BaseTool)
    )
    observed = tool_class().run(left=6, right=7)
    print('Observed invocation:', observed)
    assert observed == {'result': 42, 'success': True}


## 4. Invoke Calculator through the real MCP registration

The MCP server is default-deny. The cell temporarily enables and allowlists only `Calculator_Tool`, calls the actual registered MCP operation, and restores the environment afterward. It skips explicitly if the MCP SDK is not installed.

In [ ]:
import asyncio
import importlib.util
import os

if importlib.util.find_spec('mcp') is None:
    print("SKIPPED: install the MCP SDK with `python -m pip install -e '.[mcp]'`.")
else:
    from opentools.mcp_server import create_server

    keys = [
        'OPENTOOLS_MCP_ALLOWED_TOOLS',
        'OPENTOOLS_MCP_ALLOW_TOOL_CALLS',
        'OPENTOOLS_MCP_MAX_RISK',
    ]
    previous = {key: os.environ.get(key) for key in keys}
    try:
        os.environ['OPENTOOLS_MCP_ALLOWED_TOOLS'] = 'Calculator_Tool'
        os.environ['OPENTOOLS_MCP_ALLOW_TOOL_CALLS'] = '1'
        os.environ['OPENTOOLS_MCP_MAX_RISK'] = 'low'
        server = create_server()
        registered = server._tool_manager._tools
        print('Registered MCP operations:', sorted(registered))
        observed = await registered['call_opentool'].fn(
            'Calculator_Tool',
            {'operation': 'add', 'values': [1, 2, 3]},
        )
        print('Observed MCP result:', observed)
        assert observed['status'] == 'completed'
        assert observed['result']['result'] == '6'
    finally:
        for key, value in previous.items():
            if value is None:
                os.environ.pop(key, None)
            else:
                os.environ[key] = value


To test the transport from another MCP application, start the server separately:

```bash
OPENTOOLS_MCP_ALLOWED_TOOLS=Calculator_Tool \
OPENTOOLS_MCP_ALLOW_TOOL_CALLS=1 \
opentools-mcp --transport stdio
```

For local Streamable HTTP, use `opentools-mcp --transport streamable-http --host 127.0.0.1 --port 8000`.

## 5. Optional DSPy adapter

This creates a real `dspy.Tool` and invokes the real Calculator implementation.

In [ ]:
if importlib.util.find_spec('dspy') is None:
    print("SKIPPED: install DSPy with `python -m pip install -e '.[dspy]'`.")
else:
    from opentools.integrations.dspy import as_dspy_tool

    calculator = as_dspy_tool('Calculator_Tool')
    observed = calculator(operation='multiply', values=[6, 7])
    print('Observed DSPy result:', observed)
    assert observed['result'] == '42'


## 6. Optional contribution WebUI construction

This verifies that the real Gradio application can be constructed. It does not launch a public server or execute uploaded code.

In [ ]:
if importlib.util.find_spec('gradio') is None:
    print("SKIPPED: install Gradio with `python -m pip install -e '.[webui]'`.")
else:
    from opentools.webui import build_app

    app = build_app()
    print(f'Constructed WebUI: {type(app).__module__}.{type(app).__name__}')


## 7. Opt-in real LLM judge

The judge is intentionally disabled by default because it may require credentials and incur provider cost. Set `RUN_LLM_JUDGE=1` and configure the selected model provider before running this cell. A missing credential or provider error is reported as a real failure; no fallback verdict is generated.

In [ ]:
from opentools.evaluation import build_tool_card, judge_evaluation_report
from opentools.inventory import _load_tool, discover_tool_sources

if os.getenv('RUN_LLM_JUDGE') != '1':
    print('NOT RUN: set RUN_LLM_JUDGE=1 and real provider credentials to request a judge verdict.')
else:
    discovered = discover_tool_sources(tools_root)
    item = discovered['Calculator_Tool']
    tool = _load_tool('Calculator_Tool', item['source'])
    static = inspect_source(item['source'])
    report = {
        'inspection': static,
        'tool_card': build_tool_card(tool, static, {'status': 'not_run'}),
    }
    model = os.getenv('OPENTOOLS_JUDGE_MODEL', 'gpt-4o-mini')
    verdict = judge_evaluation_report(report, model=model)
    print(verdict)


## Recurring evaluation commands

To run a real low-risk evaluation and update the generated inventory locally:

```bash
opentools evaluate-all --tools Calculator_Tool --max-risk low --discard-raw-results
```

The repository's weekly GitHub workflow performs the configured refresh on an automation branch and opens or updates a pull request instead of writing directly to `main`.